# 🎭 Teddy Face Swap — GPU

**One tap:** press the ▶ **play button** on the left edge of the box below.

Then **wait about 4–5 minutes** (it downloads the AI + the face model). When it's ready, a **blue web link** appears at the very bottom — **tap that link on your phone**, then *Start Camera* → *Load Model*.

If something goes wrong it prints a line starting with **`STOP:`** — screenshot it and send it to me.

⚠️ Keep this tab open while you use it. The box keeps spinning on purpose after the link shows — that's normal, leave it.

In [ ]:
# ▶ TAP THE PLAY BUTTON ON THE LEFT OF THIS BOX, then wait ~4-5 min for a web link.
import subprocess, sys, os, time, re, urllib.request

def stop(msg):
    print('\n'+'='*60+'\n'+msg+'\n'+'='*60, flush=True)
    raise SystemExit

def sh(cmd, shell=False):
    return subprocess.run(cmd, shell=shell).returncode

# 1) GPU turned on?
o = subprocess.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'], capture_output=True, text=True)
if o.returncode or not o.stdout.strip():
    stop('STOP: no GPU is on. Tap  Runtime > Change runtime type > T4 GPU > Save,  then press play on this box again.')
print('GPU:', o.stdout.strip(), flush=True)

# 2) Download the app code
REPO='https://github.com/oluwacoded/Deepfakcall'; BRANCH='main'
if not os.path.isdir('/content/Deepfakcall'):
    if sh(['git','clone','--depth','1','-b',BRANCH,REPO,'/content/Deepfakcall']):
        stop('STOP: could not download the app. Check the connection and press play again.')
os.chdir('/content/Deepfakcall')
if not (os.path.exists('web_server.py') and os.path.exists('templates/index.html')):
    stop('STOP: app files missing after download. Send the agent this output.')
print('code ready', flush=True)

# 3) Install GPU dependencies (remove conflicting builds first)
subprocess.run([sys.executable,'-m','pip','-q','uninstall','-y',
                'onnxruntime','onnxruntime-gpu',
                'opencv-python','opencv-contrib-python','opencv-python-headless'])
print('installing dependencies, ~2 min...', flush=True)
if sh([sys.executable,'-m','pip','-q','install','-r','requirements-gpu.txt']):
    stop('STOP: installing dependencies failed. Send the agent this whole output.')
if sh('wget -q -O /tmp/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /tmp/cloudflared', shell=True) or not os.path.exists('/tmp/cloudflared'):
    stop('STOP: could not download the link tool. Send the agent this output.')
print('dependencies installed', flush=True)

# 4) Confirm the GPU is usable by the AI runtime
import onnxruntime as ort
if 'CUDAExecutionProvider' not in ort.get_available_providers():
    stop('STOP: the AI runtime cannot see the GPU. Copy this whole output to the agent to fix the versions.')
print('GPU ready for AI', flush=True)

# 5) Download the Amber Song face model (~718 MB)
os.makedirs('userdata/models', exist_ok=True)
mp='userdata/models/Amber_Song.dfm'
murl='https://github.com/iperov/DeepFaceLive/releases/download/AMBER_SONG/Amber_Song.dfm'
MIN=700_000_000
mok=lambda: os.path.exists(mp) and os.path.getsize(mp)>=MIN
print('downloading face model, ~2 min...', flush=True)
for attempt in range(3):
    if mok(): break
    subprocess.run(['wget','-q','-c','-O',mp,murl])
if not mok():
    stop('STOP: face model download did not finish. Press play again; if it keeps failing tell the agent.')
print('face model ready', flush=True)

# 6) Start the app, wait until it is healthy, then show a public link
os.environ['SESSION_SECRET']='colab-'+os.urandom(8).hex()
def tail(f,n=25):
    try: return ''.join(open(f).readlines()[-n:])
    except Exception: return '(no '+f+' yet)'
srv=subprocess.Popen([sys.executable,'web_server.py'], stdout=open('server.log','w'), stderr=subprocess.STDOUT)
print('starting the app...', flush=True)
ready=False
for _ in range(60):
    time.sleep(2)
    if srv.poll() is not None:
        print('SERVER LOG:\n'+tail('server.log')); stop('STOP: the app crashed on startup. Send the agent the SERVER LOG above.')
    try:
        urllib.request.urlopen('http://127.0.0.1:5000/api/models', timeout=3); ready=True; break
    except Exception: pass
if not ready:
    print('SERVER LOG:\n'+tail('server.log')); stop('STOP: the app did not start in time. Send the agent the SERVER LOG above.')
print('app is running', flush=True)
tun=subprocess.Popen(['/tmp/cloudflared','tunnel','--url','http://localhost:5000','--no-autoupdate'], stdout=open('tunnel.log','w'), stderr=subprocess.STDOUT)
url=None
for _ in range(45):
    time.sleep(2)
    mm=re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', tail('tunnel.log',200))
    if mm: url=mm.group(0); break
print('\n'+'='*60)
if url:
    print('  OPEN THIS ON YOUR PHONE (tap it):\n')
    print('     '+url+'\n')
    print('  then: Start Camera  ->  Load Model  (Amber Song is ready)')
    print('  keep this tab open while you use it.')
else:
    print('TUNNEL LOG:\n'+tail('tunnel.log'))
    print('  link not ready - send the agent the TUNNEL LOG above.')
print('='*60, flush=True)

# 7) keep the session awake (this box keeps running on purpose - leave it)
while True:
    time.sleep(30)
    try: print(tail('server.log',4), flush=True)
    except Exception: pass